# 第11章：目标设定与迭代（Goal Setting & Iteration）

## 概念

**目标设定与迭代** = 设定目标 → 生成 → 评估是否达标 → 不达标则改进 → 重复

```
设定目标 → 生成初稿 → 评估(是否达标?) → 否 → 改进 → 评估...
                              ↓ 是
                           输出结果
```

## 与第4章反思的区别

| | 第4章 反思 | 第11章 目标迭代 |
|--|-----------|----------------|
| **重点** | 找出错误/不足 | 对照目标检查 |
| **标准** | 通用质量 | 明确的目标列表 |
| **终止** | 固定次数 | 目标全部达成 |

## 代码演示

设定写作目标，让 LLM 迭代改进直到所有目标达成

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

print("MiMo 模型初始化完成！")

MiMo 模型初始化完成！


## Step 1：设定目标

为写作任务设定明确的目标列表

In [4]:
# 定义写作目标
goals = [
    "故事字数在100-200字之间",
    "包含一个意外结局",
    "语言简洁生动",
    "主题围绕'信任'"
]

print("=== 写作目标 ===")
for i, goal in enumerate(goals, 1):
    print(f"  {i}. {goal}")

=== 写作目标 ===
  1. 故事字数在100-200字之间
  2. 包含一个意外结局
  3. 语言简洁生动
  4. 主题围绕'信任'


## Step 2：生成初稿

根据目标生成初始故事

In [5]:
# 生成故事的提示词
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个作家。请根据以下目标写一个故事：

目标：
{goals}

请直接输出故事内容。"""),
    ("user", "请写一个故事。")
])

# 创建链
generate_chain = generate_prompt | llm | StrOutputParser()

# 生成初稿
goals_text = "\n".join([f"- {g}" for g in goals])
draft = generate_chain.invoke({"goals": goals_text})

print("=== 初稿 ===")
print(draft)

=== 初稿 ===
老陈把存了半辈子的铁盒交给阿强时，手在抖。盒里是给女儿治病的救命钱，阿强是他看着长大的邻居，拍胸脯保证明天就送去省城医院。

第二天，阿强没出现。第三天，老陈在他紧锁的门前等到天黑。街坊摇头：“早搬走了，听说欠了一屁股债。”

老陈瘫坐在地。一周后，女儿却从省城打来电话，哭着说手术很成功，有个年轻人垫付了全部费用，只留了张字条：“陈叔，对不起，我偷了您的信任去救急。钱一定还。”

字条背面，是皱巴巴的借条，借款人是阿强。老陈这才想起，阿强母亲当年病重时，自己也曾悄悄塞过一个铁盒。


## Step 3：评估是否达标

检查故事是否满足所有目标

In [6]:
# 评估提示词
evaluate_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个评审员。请检查故事是否满足以下目标：

目标：
{goals}

故事：
{story}

对每个目标，输出：
- 达成 / 未达成：原因

最后总结：全部达成 / 未达成"""),
    ("user", "请评估故事。")
])

# 创建链
evaluate_chain = evaluate_prompt | llm | StrOutputParser()

# 评估初稿
evaluation = evaluate_chain.invoke({"goals": goals_text, "story": draft})

print("=== 评估结果 ===")
print(evaluation)

=== 评估结果 ===
### 目标评估：

- **目标1：故事字数在100-200字之间**  
  达成：故事字数约170字，符合100-200字的范围要求。

- **目标2：包含一个意外结局**  
  达成：结局意外，读者最初以为阿强偷钱，但实际揭示阿强用钱救了老陈女儿并留下借条，体现了反转。

- **目标3：语言简洁生动**  
  达成：语言简洁直接，如“手在抖”、“瘫坐在地”等描述生动，情节推进流畅。

- **目标4：主题围绕'信任'**  
  达成：主题围绕信任展开，从老陈对阿强的信任、信任被考验，到最终信任得到回报，核心明确。

### 总结：
全部达成


## Step 4：迭代改进

如果未达成目标，根据评估结果改进故事，直到全部达成或达到最大次数

In [7]:
# 改进故事的提示词
improve_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个作家。请根据评估结果改进故事，确保满足所有目标。

目标：
{goals}

当前故事：
{current_story}

评估结果：
{evaluation}

请输出改进后的故事。"""),
    ("user", "请改进故事。")
])

# 创建链
improve_chain = improve_prompt | llm | StrOutputParser()

# 迭代改进函数
def iterate_until_goals(goals, max_iterations=3):
    goals_text = "\n".join([f"- {g}" for g in goals])
    
    # 生成初稿
    story = generate_chain.invoke({"goals": goals_text})
    print("\n=== 初稿 ===")
    print(story)
    
    for i in range(max_iterations):
        # 评估
        evaluation = evaluate_chain.invoke({"goals": goals_text, "story": story})
        print(f"\n=== 评估 (第{i+1}次) ===")
        print(evaluation)
        
        # 检查是否全部达成
        if "全部达成" in evaluation:
            print(f"\n✅ 所有目标在第{i+1}次评估时达成！")
            return story
        
        # 改进故事
        story = improve_chain.invoke({
            "goals": goals_text,
            "current_story": story,
            "evaluation": evaluation
        })
        print(f"\n=== 改进稿 (第{i+1}次) ===")
        print(story)
    
    print(f"\n⚠️ 达到最大迭代次数({max_iterations})，返回最新版本。")
    return story

# 运行迭代
print("\n" + "="*50)
print("开始目标迭代")
print("="*50)

final_story = iterate_until_goals(goals, max_iterations=3)

print("\n" + "="*50)
print("最终故事")
print("="*50)
print(final_story)


开始目标迭代

=== 初稿 ===
将军把最后一块压缩饼干掰成两半，递给年轻的士兵：“吃吧，明天就能到补给站了。”

士兵看着将军干裂的嘴唇，摇头：“您三天没吃东西了。”

“这是命令。”将军把饼干塞进他手里，转身望向漆黑的山谷。

黎明时分，侦察兵带回消息：补给站已被敌军占领。士兵握紧枪，想起那半块饼干——将军早知道没有补给站了。

他冲向将军的帐篷，却只找到一张地图，上面标着一条隐秘的撤退路线，终点是安全区。地图背面写着：“活下去，这是最后的信任。”

远处传来炮火声，将军独自走向了相反的方向。

=== 评估 (第1次) ===
### 目标评估：

1. **故事字数在100-200字之间**  
   - 达成：故事总字数约为110字，符合100-200字的要求。

2. **包含一个意外结局**  
   - 达成：结局中，士兵发现将军早已知道补给站被占，却通过地图提供撤退路线，而将军独自走向相反方向（可能牺牲自己），这构成了意外转折。

3. **语言简洁生动**  
   - 达成：语言简洁明了，如“干裂的嘴唇”、“漆黑的山谷”、“炮火声”等描述生动，增强了故事的画面感和情感张力。

4. **主题围绕'信任'**  
   - 达成：故事核心围绕信任展开，将军通过饼干和地图表达对士兵的信任（“活下去，这是最后的信任”），士兵则从信任命令到理解将军的深意，主题贯穿始终。

### 总结：全部达成

✅ 所有目标在第1次评估时达成！

最终故事
将军把最后一块压缩饼干掰成两半，递给年轻的士兵：“吃吧，明天就能到补给站了。”

士兵看着将军干裂的嘴唇，摇头：“您三天没吃东西了。”

“这是命令。”将军把饼干塞进他手里，转身望向漆黑的山谷。

黎明时分，侦察兵带回消息：补给站已被敌军占领。士兵握紧枪，想起那半块饼干——将军早知道没有补给站了。

他冲向将军的帐篷，却只找到一张地图，上面标着一条隐秘的撤退路线，终点是安全区。地图背面写着：“活下去，这是最后的信任。”

远处传来炮火声，将军独自走向了相反的方向。


## 目标迭代流程

```
┌─────────────────────────────────────────────────────────┐
│                目标设定与迭代流程                         │
├─────────────────────────────────────────────────────────┤
│  设定目标（明确、可衡量）                                 │
│     ↓                                                   │
│  生成初稿                                                │
│     ↓                                                   │
│  评估（对照目标检查）                                     │
│     ↓                                                   │
│  全部达成？ ──是──→ 输出结果                              │
│     │                                                   │
│     否                                                  │
│     ↓                                                   │
│  改进故事                                                │
│     ↓                                                   │
│  回到评估                                                │
└─────────────────────────────────────────────────────────┘
```

## 目标设定原则

| 原则 | 说明 | 示例 |
|------|------|------|
| **明确** | 不模糊 | "100-200字" 而非 "不要太长" |
| **可衡量** | 能判断是否达成 | "包含意外结局" 而非 "有趣" |
| **有限** | 目标数量适中 | 3-5 个目标 |
| **不冲突** | 目标之间兼容 | 不要同时要求 "简短" 和 "详细" |

## 与其他模式的关系

| 第4章 反思 | 第6章 规划 | 第9章 适应 | 第11章 目标迭代 |
|-----------|----------|----------|---------------|
| 生成→批评→改进 | 计划→执行 | 根据反馈改进 | 目标→评估→改进 |
| 通用质量改进 | 结构化执行 | 环境/反馈驱动 | 目标驱动的迭代 |

## 实际应用场景

- **代码生成**：设定功能、性能、可读性目标，迭代改进
- **文案创作**：设定字数、风格、情感目标
- **方案设计**：设定成本、可行性、效果目标
- **翻译**：设定准确度、流畅度、专业术语目标